# Sankey Diagram Assignment

## Overview

This assignment recreates a Sankey-style visualization showing how five stroke risk factors change over time from **1990 to 2010**. The visualization uses five time points and five risk factors:

* Hypertension
* Diabetes
* Smoking
* Hypercholesterolemia
* Obesity

The main challenge was that the desired graphic is not a standard Sankey diagram produced by an existing charting function. Instead, the visualization consists of **stacked rounded rectangles ("plateaus") connected by sloped polygonal bands**. Therefore, the solution required constructing the geometry of the chart manually and then using `ggplot2` to render it.

The goal of the code was not only to display the data, but to reproduce the visual structure of the reference graphic while keeping the geometry, spacing, colors, labels, and layering under explicit control.

---

## 1. Problem Definition

The starting data contains a value for each risk factor at five different time points. The years are stored separately as:

```r
years <- c(1990, 1995, 2000, 2005, 2010)
```

The plotting system uses `yr = 1:5` as the x-coordinate for the five time positions, while the actual calendar years are added as labels at the top of the chart.

This separation solves an important layout problem: using simple positions `1` through `5` makes it easier to control the width of the plateaus and the distance between adjacent years without having to work with the much larger numerical differences between calendar years.

---

## 2. Understanding the Visual Structure

The diagram can be broken into three main geometric components:

1. **Stacked bands at each year**
2. **Sloped connectors between consecutive years**
3. **Text labels and chart annotations**

The bands represent the magnitude of each risk factor. Their vertical position changes from year to year, while their colored regions remain associated with the same risk factor.

The code therefore treats the chart as a geometry problem rather than relying on a pre-built Sankey plotting function.

---

## 3. Preparing the Data

The five risk factors are combined into one data frame using `rbind()`. Each risk has five observations, one for each time position.

The values are retained in their original form because they determine the height of the visual bands.

For example, hypertension starts at `0.654` and decreases to `0.509`, while obesity increases from `0.001` to `0.115`.

This creates an important visualization problem: some values are extremely small compared with others. If the raw values were used directly as geometric heights, the smallest bands could become almost invisible.

---

## 4. Creating a Usable Stack

The first major problem-solving step is creating the vertical stack for every year.

```r
mutate(h = value + MIN_H)
```

adds a small minimum height to every band. This means that even a very small value receives enough visual space to remain visible. The code describes this as "raising the platform."

The data is then grouped by year:

```r
group_by(yr)
```

because the vertical arrangement needs to be calculated independently for each time point.

Within each year, the risks are sorted from smallest to largest. This determines the order of the bands in the stack.

---

## 5. Calculating Band Positions

Once the order is known, the code calculates three important vertical coordinates:

* `ymin` — bottom of the band
* `ymax` — top of the band
* `ymid` — center of the band

The key calculation is:

```r
ymin = cumsum(h) - h + GAP * (rk - 1)
```

The cumulative height determines how far up the stack the current band should begin. The subtraction of `h` moves the position back to the bottom of the current band.

The additional gap term separates adjacent bands visually.

Finally:

```r
ymax = ymin + h
ymid = (ymin + ymax) / 2
```

determines the top and center of each band.

This calculation is the foundation of the entire visualization because the same `ymin` and `ymax` coordinates are later used to construct both the plateaus and the connectors.

---

## 6. Creating Rounded Plateaus

A normal `geom_rect()` would be sufficient for ordinary rectangles, but the reference visualization uses rounded corners.

To solve this, the code creates a custom function called `rrect()`.

```r
rrect <- function(xmin, xmax, ymin, ymax, rx, ry, n = 8)
```

Instead of drawing the rectangle directly, the function calculates points along four quarter-circle/ellipse arcs and joins them together.

The advantage of constructing the rectangle from points is that the resulting shape can be passed to:

```r
geom_polygon()
```

This gives precise control over the appearance of each band.

The function also protects very thin bands by limiting the vertical corner radius:

```r
ry <- min(ry, (ymax - ymin) / 2)
```

Without this limitation, rounded corners could overlap when the band is very short.

---

## 7. Generating Every Plateau

The next step applies the rounded rectangle function to every row of the stacked data.

For each band, the code:

1. Retrieves the current row.
2. Centers a rectangle around the year.
3. Uses the calculated `ymin` and `ymax`.
4. Applies the corner-radius settings.
5. Assigns an ID to the shape.
6. Stores the associated risk factor.

This produces the `plateaus` data frame.

The important design decision here is to create the geometry as data before plotting it. This separates **geometry construction** from **visual rendering**, making the chart easier to control and troubleshoot.

---

## 8. Creating the Sloped Connectors

Because the bands can change both size and vertical position, the connector must connect:

* the bottom of the current band,
* the top of the current band,
* the top of the next band,
* the bottom of the next band.

To prepare for this, the data is grouped by risk factor and ordered chronologically. The `lead()` function obtains the corresponding vertical positions in the following year.

The final year is removed because it does not have a following year to connect to.

---

## 9. Constructing Each Slope as a Polygon

Each connector is represented by four points.

The left side is positioned just outside the current plateau:

```r
xL <- r$yr + HALF_WIDTH + WHITE_SPACE
```

The right side is positioned just before the next plateau:

```r
xR <- r$yr + 1 - HALF_WIDTH - WHITE_SPACE
```

The vertical coordinates are then:

```r
y = c(r$ymin, r$ymax, r$ymax2, r$ymin2)
```

These four coordinates create a quadrilateral whose width and height change from one year to the next.

---

## 10. Controlling Layering

The diagram contains many overlapping polygons, so drawing order matters.

The code defines:

```r
DRAW_ORDER <- c(
  "Hypertension",
  "Diabetes",
  "Smoking",
  "Obesity",
  "Hypercholesterolemia"
)
```

and then uses this order when arranging the slope data.

This ensures that connectors are drawn in a predictable order rather than relying on the original row order of the data.

---

## 11. Making Small Values Visible

The code addresses this using:

```r
MIN_H <- 0.015
```

and calculates:

```r
h = value + MIN_H
```

The original `value` remains available for labeling, while the adjusted `h` is used for the graphical height.

This allows the visualization to communicate the original values while preventing extremely small categories from disappearing completely.

---

## 12. Formatting the Labels

The values are formatted separately from the geometry.

```r
vlab <- stack %>%
  mutate(txt = sprintf("%.2f", ...))
```

This produces labels with exactly two decimal places.

The labels are placed at:

```r
x = yr
y = ymid
```

so that each value appears in the vertical center of its corresponding band.

The risk-factor names are taken only from the first year:

```r
catlab <- stack %>% filter(yr == 1)
```

This prevents the category names from being repeated five times across the diagram.

---

## 13. Reproducing the Overall Layout

The chart uses explicit x- and y-axis limits:

```r
XLIM <- c(-0.277, 5.768)
YLIM <- c(-0.082, 2.017)
```

These limits provide room for the category labels on the left, the title above the chart, and the stacked bands themselves.

The use of:

```r
coord_cartesian(..., expand = FALSE, clip = "off")
```

allows the geometry to extend into the label area without being clipped.

This is useful because the chart is being recreated according to a specific visual layout rather than using automatically generated plot margins.

---

## 14. Building the Final Plot

The final plot is constructed in layers.

First, the sloped connectors are drawn:

```r
geom_polygon(data = slopes, ...)
```

Then the plateaus are drawn:

```r
geom_polygon(data = plateaus, ...)
```

This creates the main Sankey structure.

The text values are then added in white and bold so that they remain readable against the colored bands.

The category names are added on the left, followed by the year labels and title.

Finally, `scale_fill_manual()` applies the specified colors to each risk factor.

---

## 18. Overall Workflow

The complete problem-solving workflow can be summarized as:

```text
Raw risk-factor data
        ↓
Assign time positions
        ↓
Calculate visual band heights
        ↓
Sort bands within each year
        ↓
Calculate ymin / ymax / ymid
        ↓
Create rounded plateau polygons
        ↓
Find corresponding bands in the next year
        ↓
Create four-point slope polygons
        ↓
Format value and category labels
        ↓
Set chart geometry and appearance
        ↓
Layer polygons and text with ggplot2
        ↓
Final Sankey-style visualization
```

This workflow makes the code easier to understand because each stage solves one specific geometric or presentation problem.

---

## 19. Conclusion

The main lesson from this assignment is that a complex-looking Sankey visualization can be constructed by breaking it into smaller geometric problems.

Rather than searching for a single function that produces the entire chart, the solution builds the visualization from:

* calculated vertical positions,
* rounded polygonal plateaus,
* four-point connector polygons,
* carefully controlled spacing,
* explicit colors,
* formatted labels, and
* manually defined layout coordinates.

The final `ggplot2` object combines these components into the completed visualization.

The approach demonstrates how data manipulation with `dplyr` and geometric construction with `ggplot2` can be combined to reproduce a customized visualization when a standard chart type does not provide sufficient control.


## Visual Results
The figure below represents the completed recreation of the Excel visualization using `ggplot2`.

The colored bands represent the five risk factors, while the connecting ribbons show how the relative values change between consecutive years. The numerical labels provide the underlying values for each risk factor and year

<div align="center">
    <img src="sankey_diagram.png" width="50%">
</div>
    

## Comparison with the Excel Chart

The recreated Sankey-style diagram was designed to match the original Excel chart as closely as possible using `ggplot2`. The main visual features reproduced include the ribbon colors, ribbon opacity, text colors, text placement, rounded shapes, and spacing between the graphical elements.

The additional whitespace versions demonstrate how the appearance of the diagram changes when the spacing between bands and connecting ribbons is removed.

<div align="center">
    <img src="Picture1.jpg" width="50%">
</div>
<div align="center">
    <img src="sankey_diagram.png" width=50%>
</div>

Comparison With the Excel Visualization
Using custom polygon geometry provides greater control over the visualization than a standard bar chart or a basic Sankey implementation.
The R implementation was designed to reproduce the major visual characteristics of the original Excel visualization.
The following aspects were specifically controlled:

| Feature | Approach |
|---|---|
| Ribbon color | Manually specified hexadecimal colors |
| Ribbon opacity | Controlled with the `alpha` parameter |
| Numerical text | Centered within each band |
| Text color | White numerical labels and black category/year labels |
| Category labels | Positioned along the left side of the plot |
| Year labels | Positioned above the yearly plateaus |
| Rounded corners | Custom polygon geometry |
| Horizontal spacing | Controlled through plateau and connector coordinates |
| Vertical spacing | Controlled through band calculations |
| Plot boundaries | Manually specified coordinate limits |



## Effect of Horizontal and Vertical White Space

The assignment specifically encouraged examining versions of the Sankey diagram with and without the horizontal and vertical white space separating the graphical elements.

Two parameters were used to control these spaces:

- `WHITE_SPACE` controls the vertical separation between adjacent bands within a given year.
- `GAP` controls the horizontal separation between a yearly plateau and its connecting ribbon.

Four versions of the visualization were considered. The first version contains both types of spacing, while the remaining versions progressively remove either the vertical spacing, horizontal spacing, or both.

| Version | `GAP` | `WHITE_SPACE` | Description |
|---|---:|---:|---|
| With both | 0.006 | 0.038 | Vertical and horizontal white space |
| No vertical | 0 | 0.038 | Bands touch within each year |
| No horizontal | 0.006 | 0 | Plateaus touch their connecting ribbons |
| No white space | 0 | 0 | Both types of spacing are removed |

These four configurations allow the visual effect of each spacing parameter to be examined independently.

### With Both Types of White Space

The original configuration uses both vertical and horizontal spacing. This provides visual separation between adjacent bands and between the plateaus and their connecting ribbons.

<div align="center">
    <img src="sankey_diagram.png" width="50%">
</div>

### Without Vertical White Space

In this version, `WHITE_SPACE` is set to zero. The bands within each year therefore touch one another vertically, while the horizontal separation between the plateaus and ribbons remains.
<div align="center">
    <img src="sankey_diagram_no_vertical.png" width="50%">
</div>

### Without Horizontal White Space

In this version, `GAP` is set to zero. The connecting ribbons meet the yearly plateaus directly, while vertical spacing between the bands remains.
<div align="center">
    <img src="sankey_diagram_no_horizontal.png" width="50%">
</div>

### Without White Space

Finally, both parameters are set to zero. The bands touch vertically and the connecting ribbons meet the plateaus directly.
<div align="center">
    <img src="sankey_diagram_no_space.png" width="50%">
</div>

### Main Files Used

The main files used for this assignment are:

| File / Item                             | Description                                                                                          |
| --------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| `Assignment No 2`                       | Assignment project and associated materials for recreating the Sankey-style diagram using `ggplot2`. |
| `notebooks_assignment_two_README.ipynb` | Jupyter Notebook/README for assignment two.                                                          |
| `Picture1.jpg`                          | Reference image used in the project.                                                                 |
| `sankey_diagram.R`                      | R script containing the code used to recreate the Sankey-style diagram.                              |
| `sankey_diagram.png`                    | Main recreated Sankey-style diagram.                                                                 |
| `sankey_diagram_no_horizontal.png`      | Sankey-style diagram without horizontal white space.                                                 |
| `sankey_diagram_no_vertical.png`        | Sankey-style diagram without vertical white space.                                                   |
| `sankey_diagram_no_space.png`           | Sankey-style diagram without horizontal or vertical white space.                                     |

